# GBM Model Walkthrough

This notebook walks through the LightGBM quantitative model end-to-end: what data it uses, how features are built, what the model learns, and how we evaluate whether it actually works.

Each section has an explanation of **why** we made a particular choice, followed by code you can run and inspect.

## Prerequisites

You need to have already run:
```
python -m src.quant_research.download   # ~30 min, downloads 12yr price history
python -m src.quant_research.features   # ~5 min, builds feature matrix
python -m src.quant_research.train --model gbm  # trains and saves artifacts
```

In [ ]:
import math
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make sure we're running from the project root
import os
if Path('src').exists():
    print('Working directory looks correct:', Path('.').resolve())
else:
    print('WARNING: Run this notebook from the marketview/ project root')

from src.quant_research.features import FEATURE_COLS

---
## Part 1: The Raw Data

We downloaded 12 years of daily OHLCV (open/high/low/close/volume) data for all ~900 S&P 500 and S&P 400 tickers using yfinance. That's roughly **3 million rows** before feature filtering.

**Why 12 years?** We need enough history to:
- Train on 10 years (roughly 2,500 trading days per ticker)
- Validate on the most recent 2 years (held out, never seen during training)
- Compute long lookback features — the longest is a 756-day (3-year) log return, so each row needs 756 days of prior data before it can even be computed

**Why S&P 500 + 400?** These are liquid, well-covered stocks. Pure survivorship bias still exists (companies that failed were removed from the index), but it's much less severe than screening on "stocks that exist today."

In [ ]:
raw = pd.read_parquet('data/quant/raw_prices.parquet')
raw['date'] = pd.to_datetime(raw['date'])

print(f"Rows:          {len(raw):,}")
print(f"Tickers:       {raw['ticker'].nunique()}")
print(f"Date range:    {raw['date'].min().date()} → {raw['date'].max().date()}")
print(f"Columns:       {list(raw.columns)}")
print()
print(raw.sample(5).to_string())

In [ ]:
# How many trading days of history does each ticker have?
days_per_ticker = raw.groupby('ticker')['date'].count().sort_values()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(days_per_ticker, bins=40, edgecolor='black')
ax.axvline(days_per_ticker.median(), color='red', linestyle='--', label=f'Median: {days_per_ticker.median():.0f} days')
ax.set_xlabel('Trading days of history per ticker')
ax.set_ylabel('Number of tickers')
ax.set_title('Price history depth across universe')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Tickers with full 12yr history (>3000 days): {(days_per_ticker > 3000).sum()}")
print(f"Tickers with <756 days (can't compute features): {(days_per_ticker < 756).sum()}")

---
## Part 2: The Feature Matrix

Raw prices aren't useful directly — a stock at $500 and a stock at $10 aren't comparable. We engineer 15 features that are either **percentages** or **log returns** so that all tickers are on the same scale.

The one exception is `log_price` (the log of the closing price), which is intentionally absolute. It captures the idea that a $500 stock is typically a larger, more established company than a $5 stock — useful long-term context the model shouldn't throw away.

### Feature categories:

| Feature | What it measures | Why it might matter |
|---|---|---|
| `log_price` | log(close) | Proxy for company size / growth stage |
| `pct_sma10/50/200` | % above/below moving average | Short/medium/long-term trend |
| `pct_ath` | % below all-time high | How far from peak (negative = below ATH) |
| `pct_time_since_ath` | % of 5-year window since ATH | How long the stock has been off its high |
| `pct_52w_low` | % above 52-week low | Where in the annual range the stock sits |
| `log_ret_5/20/60/126/252/756d` | Log return over N days | Momentum across multiple timeframes |
| `vol_20d/60d` | Annualized volatility | Risk profile |

### The target: `fwd_log_ret_20d`

For every (ticker, date) row, we compute the actual log return over the following 20 trading days:
```
fwd_log_ret_20d = log(close[t+20] / close[t])
```
This is what the model learns to predict. We use log returns (not simple %) because they're additive across time and symmetric around zero.

In [ ]:
features = pd.read_parquet('data/quant/features.parquet')
features['date'] = pd.to_datetime(features['date'])

train = features[features['split'] == 'train']
val   = features[features['split'] == 'val']

print(f"Total rows:    {len(features):,}")
print(f"Train rows:    {len(train):,}  ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Val rows:      {len(val):,}  ({val['date'].min().date()} → {val['date'].max().date()})")
print(f"Tickers:       {features['ticker'].nunique()}")
print(f"Features:      {FEATURE_COLS}")

In [ ]:
# Look at the distribution of the TARGET variable
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, split, df in [(axes[0], 'Train', train), (axes[1], 'Val', val)]:
    ax.hist(df['fwd_log_ret_20d'], bins=80, range=(-0.5, 0.5), edgecolor='none', alpha=0.7)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.axvline(df['fwd_log_ret_20d'].mean(), color='red', linestyle='--',
               label=f'Mean: {df["fwd_log_ret_20d"].mean():.3f}')
    ax.set_title(f'{split} set: 20-day forward log return')
    ax.set_xlabel('fwd_log_ret_20d')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

print("Train target stats:")
print(train['fwd_log_ret_20d'].describe().round(4))
print("\nVal target stats:")
print(val['fwd_log_ret_20d'].describe().round(4))

In [ ]:
# Inspect feature statistics side-by-side for train vs val
# These should look similar if the val period isn't a regime outlier
comparison = pd.DataFrame({
    'train_mean': train[FEATURE_COLS].mean(),
    'val_mean':   val[FEATURE_COLS].mean(),
    'train_std':  train[FEATURE_COLS].std(),
    'val_std':    val[FEATURE_COLS].std(),
}).round(4)
comparison['mean_shift'] = (comparison['val_mean'] - comparison['train_mean']).round(4)
print(comparison.to_string())

In [ ]:
# Drill into a single ticker to see what the features look like over time
TICKER = 'AAPL'  # change this to any ticker in the dataset

ticker_df = features[features['ticker'] == TICKER].sort_values('date')
print(f"{TICKER}: {len(ticker_df)} rows from {ticker_df['date'].min().date()} to {ticker_df['date'].max().date()}")
print()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(ticker_df['date'], ticker_df['close'], linewidth=1)
axes[0].set_title(f'{TICKER} close price')
axes[0].set_ylabel('Price ($)')

axes[1].plot(ticker_df['date'], ticker_df['pct_sma200'], linewidth=1, label='pct_sma200')
axes[1].plot(ticker_df['date'], ticker_df['pct_ath'], linewidth=1, label='pct_ath', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Position relative to SMA200 and all-time high')
axes[1].set_ylabel('%')
axes[1].legend()

axes[2].plot(ticker_df['date'], ticker_df['fwd_log_ret_20d'], linewidth=0.8, alpha=0.7)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title('Actual 20-day forward log return (the target)')
axes[2].set_ylabel('log return')

plt.tight_layout()
plt.show()

---
## Part 3: The Train / Validation Split

**Key design decision:** we split by time, not randomly.

- **Train**: first ~10 years (everything before the last 2 years)
- **Val**: last 2 years — the model never sees this data during training

**Why not random split?** If we randomly shuffled rows and put 20% aside for validation, rows from the same ticker and nearby dates would appear in both train and val. The model would essentially have "peeked" at the future. Time-based splits prevent any form of lookahead.

**What the val period tells us:** the last 2 years (roughly 2024–2026) is our best estimate of how the model will perform going forward. If the model learned something spurious about the 2013–2023 period that doesn't generalize, the val set exposes it.

In [ ]:
# Visualize the split cutoff
cutoff = val['date'].min()
print(f"Train: up to {train['date'].max().date()}")
print(f"Val:   from {val['date'].min().date()} to {val['date'].max().date()}")
print(f"Cutoff: {cutoff.date()}")

# Show the distribution of rows over time
monthly_counts = features.groupby([features['date'].dt.to_period('M'), 'split']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 4))
monthly_counts['train'].plot(ax=ax, color='steelblue', label='Train', alpha=0.8)
monthly_counts.get('val', pd.Series(dtype=float)).plot(ax=ax, color='orange', label='Val', alpha=0.8)
ax.axvline(str(cutoff.to_period('M')), color='red', linestyle='--', label='Split cutoff')
ax.set_title('Feature rows per month (train vs val)')
ax.set_ylabel('Rows')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 4: The GBM Model

We use **LightGBM** (`LGBMRegressor`) — a gradient boosting framework. Here's what that means in plain English:

### How gradient boosting works

1. Start with a simple guess (e.g., "predict the mean return for every stock")
2. Look at which rows were wrong and in which direction
3. Build a small decision tree that corrects those errors
4. Add that tree to the model, repeat 500 times

Each tree is weak on its own, but 500 trees stacked together can capture complex nonlinear relationships — like "when a stock is far below its ATH *and* has been there for a long time *and* volatility is high, the expected return is elevated."

### Key hyperparameters we used

| Parameter | Value | What it controls |
|---|---|---|
| `n_estimators` | 500 | Number of trees. More = more complex, slower |
| `max_depth` | 6 | How deep each tree can grow. Limits overfitting |
| `learning_rate` | 0.05 | How much each tree corrects errors. Lower = more conservative |
| `subsample` | 0.8 | Each tree only sees 80% of rows. Adds diversity |
| `colsample_bytree` | 0.8 | Each tree only sees 80% of features. Adds diversity |

We use `StandardScaler` to normalize features before training (mean=0, std=1). LightGBM doesn't technically need this — unlike KNN, it's tree-based and scale-invariant — but we do it anyway so that inference is consistent across all models.

In [ ]:
# Load the trained artifacts
artifacts_dir = Path('data/quant/artifacts/gbm')

with open(artifacts_dir / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open(artifacts_dir / 'model.pkl', 'rb') as f:
    model = pickle.load(f)

print(f"Model type: {type(model).__name__}")
print(f"Number of trees: {model.n_estimators_}")
print(f"Number of features: {model.n_features_in_}")
print(f"\nScaler mean (first 5 features):")
for feat, mean, std in zip(FEATURE_COLS[:5], scaler.mean_[:5], scaler.scale_[:5]):
    print(f"  {feat:<22} mean={mean:>8.3f}  std={std:>8.3f}")

---
## Part 5: Feature Importances

LightGBM tracks how many times each feature was used to split data across all 500 trees. More splits = the model found that feature more useful for reducing prediction error.

**Important caveat:** high importance doesn't mean a feature is causal. It means the model found it statistically useful for predicting 20-day returns in the training data. A feature could be important because it's genuinely predictive, or because it correlates with an omitted variable, or because it picked up noise.

In [ ]:
importances = sorted(
    zip(FEATURE_COLS, model.feature_importances_),
    key=lambda x: x[1],
    reverse=True
)

feats, imps = zip(*importances)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2196F3' if i < 5 else '#90CAF9' for i in range(len(feats))]
bars = ax.barh(range(len(feats)), imps, color=colors)
ax.set_yticks(range(len(feats)))
ax.set_yticklabels(feats)
ax.invert_yaxis()
ax.set_xlabel('Feature importance (split count across all trees)')
ax.set_title('GBM Feature Importances')
plt.tight_layout()
plt.show()

print("Top 5 features:")
for feat, imp in importances[:5]:
    print(f"  {feat:<22} {imp:>6}  ({imp/sum(imps)*100:.1f}% of total splits)")

In [ ]:
# What do the top features actually mean in plain terms?
# Let's look at how the top feature correlates with forward returns in the val set

top_feature = importances[0][0]
print(f"Top feature: {top_feature}")
print()

# Bin the top feature and compute mean forward return per bin
val_copy = val.copy()
val_copy['feature_bin'] = pd.qcut(val_copy[top_feature], q=10, labels=False, duplicates='drop')

bin_stats = val_copy.groupby('feature_bin')['fwd_log_ret_20d'].agg(['mean', 'count', 'std']).round(5)
bin_edges = val_copy.groupby('feature_bin')[top_feature].agg(['min', 'max']).round(2)
bin_stats['range'] = bin_edges.apply(lambda r: f"{r['min']} to {r['max']}", axis=1)

print(f"Mean 20d forward return by decile of {top_feature} (val set):")
print(bin_stats[['range', 'mean', 'count']].to_string())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(bin_stats)), bin_stats['mean'], color='steelblue')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(bin_stats)))
ax.set_xticklabels([f'D{i+1}' for i in range(len(bin_stats))])
ax.set_xlabel(f'{top_feature} decile (D1=lowest, D10=highest)')
ax.set_ylabel('Mean fwd_log_ret_20d (val set)')
ax.set_title(f'Forward return by {top_feature} decile')
plt.tight_layout()
plt.show()

---
## Part 6: How We Evaluate — The Simulated Portfolio

We don't just look at prediction error (RMSE). We simulate what actually happens if you use the model to make trades.

### The evaluation loop

Every 20 trading days (non-overlapping — so picks from one period don't overlap with the next):

1. Score all tickers on that date using the GBM model
2. Buy the top 20 by predicted return, equal-weight
3. Hold for exactly 20 trading days
4. Record the actual return

Over 2 years that's 26 evaluation periods. We then compute:
- **AvgRet**: mean 20-day log return of our picks
- **SPYRet**: what SPY returned over the same 20-day period
- **Excess**: our return minus SPY — pure alpha
- **HitRate**: what % of our 20 picks had positive returns
- **Sharpe**: excess return / std(excess return) × √(252/20) — annualized

In [ ]:
# Reproduce the evaluation manually so we can inspect each period

X_val = val[FEATURE_COLS].values
X_val_scaled = scaler.transform(X_val)
val = val.copy()
val['predicted'] = model.predict(X_val_scaled)

_FORWARD_DAYS = 20
_TOP_N = 20

all_dates = sorted(val['date'].unique())
eval_dates = all_dates[::_FORWARD_DAYS]

spy_returns = (
    val[val['ticker'] == 'SPY']
    .set_index('date')['fwd_log_ret_20d']
    .to_dict()
)

period_results = []

for d in eval_dates:
    day_df = val[(val['date'] == d) & (val['ticker'] != 'SPY')]
    if len(day_df) < _TOP_N:
        continue
    top = day_df.nlargest(_TOP_N, 'predicted')
    actual = top['fwd_log_ret_20d'].dropna()
    if len(actual) < _TOP_N // 2:
        continue
    spy_ret = spy_returns.get(d, float('nan'))
    port_ret = actual.mean()
    period_results.append({
        'date': d,
        'port_ret': port_ret,
        'spy_ret': spy_ret,
        'excess': port_ret - spy_ret if not math.isnan(spy_ret) else float('nan'),
        'hit_rate': (actual > 0).mean(),
        'n_picks': len(actual),
        'picks': ', '.join(top['ticker'].tolist()),
    })

results_df = pd.DataFrame(period_results).set_index('date')
print(f"Evaluation periods: {len(results_df)}")
print(results_df[['port_ret', 'spy_ret', 'excess', 'hit_rate']].round(4).to_string())

In [ ]:
# Summary metrics
excess = results_df['excess'].dropna()
periods_per_year = 252 / _FORWARD_DAYS
sharpe = excess.mean() / excess.std() * math.sqrt(periods_per_year)

print("=" * 50)
print("GBM Validation Results (last 2 years)")
print("=" * 50)
print(f"  Periods:          {len(results_df)}")
print(f"  Avg portfolio ret: {results_df['port_ret'].mean():.4f}  ({results_df['port_ret'].mean()*100:.2f}%)")
print(f"  Avg SPY ret:       {results_df['spy_ret'].mean():.4f}  ({results_df['spy_ret'].mean()*100:.2f}%)")
print(f"  Avg excess ret:    {excess.mean():.4f}  ({excess.mean()*100:.2f}%)")
print(f"  Hit rate:          {results_df['hit_rate'].mean():.1%}")
print(f"  Sharpe (ann.):     {sharpe:.3f}")
print("=" * 50)
print()
print("Interpretation:")
print(f"  Every 20 trading days, our top-20 picks returned {excess.mean()*100:.1f}% more")
print(f"  than SPY over the same period.")
print(f"  That's roughly {excess.mean() * periods_per_year * 100:.1f}% annualized alpha (naive estimate).")

In [ ]:
# Plot cumulative returns: portfolio vs SPY
cumulative_port = (1 + results_df['port_ret']).cumprod()
cumulative_spy  = (1 + results_df['spy_ret']).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(results_df.index, cumulative_port.values, label='GBM Top-20', color='steelblue', linewidth=2)
axes[0].plot(results_df.index, cumulative_spy.values, label='SPY', color='orange', linewidth=2, linestyle='--')
axes[0].set_title('Cumulative return: GBM top-20 vs SPY (val set, 20-day periods)')
axes[0].set_ylabel('Growth of $1')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(results_df.index, results_df['excess'].values,
            color=['#2196F3' if x >= 0 else '#EF5350' for x in results_df['excess']],
            width=15)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Excess return per period (portfolio − SPY)')
axes[1].set_ylabel('Log return difference')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 7: Inspecting Individual Picks

Here we can look at what the model was buying on any given evaluation date, and see whether those picks actually worked out.

In [ ]:
# Look at picks for a specific evaluation date
# Change this to any date in the results_df index
INSPECT_DATE = results_df.index[-1]  # most recent eval date

day_df = val[(val['date'] == INSPECT_DATE) & (val['ticker'] != 'SPY')]
top20 = day_df.nlargest(20, 'predicted')[[
    'ticker', 'predicted', 'fwd_log_ret_20d',
    'pct_sma200', 'pct_ath', 'log_ret_252d', 'vol_20d'
]].reset_index(drop=True)

top20['predicted'] = top20['predicted'].round(4)
top20['fwd_log_ret_20d'] = top20['fwd_log_ret_20d'].round(4)
top20['pct_sma200'] = top20['pct_sma200'].round(1)
top20['pct_ath'] = top20['pct_ath'].round(1)
top20['log_ret_252d'] = top20['log_ret_252d'].round(3)
top20['vol_20d'] = top20['vol_20d'].round(3)
top20['beat_spy'] = top20['fwd_log_ret_20d'] > spy_returns.get(INSPECT_DATE, 0)

spy_ret_on_date = spy_returns.get(INSPECT_DATE, float('nan'))
print(f"Date: {INSPECT_DATE.date()}  |  SPY return: {spy_ret_on_date:.4f}  |  Portfolio return: {results_df.loc[INSPECT_DATE, 'port_ret']:.4f}")
print()
print(top20.to_string(index=False))

In [ ]:
# For any pick, compare its predicted vs actual return
fig, ax = plt.subplots(figsize=(10, 5))

# Scatter: predicted vs actual for all val-period top-20 picks
all_picks = []
for d in eval_dates:
    day_df = val[(val['date'] == d) & (val['ticker'] != 'SPY')]
    if len(day_df) < _TOP_N:
        continue
    top = day_df.nlargest(_TOP_N, 'predicted')
    all_picks.append(top[['ticker', 'predicted', 'fwd_log_ret_20d']].assign(date=d))

all_picks_df = pd.concat(all_picks, ignore_index=True).dropna()

ax.scatter(all_picks_df['predicted'], all_picks_df['fwd_log_ret_20d'],
           alpha=0.2, s=10, color='steelblue')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('GBM predicted 20d log return')
ax.set_ylabel('Actual 20d log return')
ax.set_title(f'Predicted vs Actual (all val-period top-20 picks, n={len(all_picks_df)})')
ax.set_xlim(-0.05, 0.25)
ax.set_ylim(-0.6, 0.6)

# Correlation
corr = all_picks_df[['predicted', 'fwd_log_ret_20d']].corr().iloc[0, 1]
ax.text(0.05, 0.92, f'Correlation: {corr:.3f}', transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

print(f"Note: low correlation ({corr:.3f}) is normal for stock return models.")
print("What matters is that the tail of high-predicted stocks outperforms on average.")

---
## Part 8: What the Model Is Actually Buying

Let's look at the feature profile of stocks that the GBM consistently selects. This tells us what market condition the model believes precedes good returns.

In [ ]:
# Compare feature means: top-20 picks vs all val stocks vs bottom-20

all_picks_with_features = []
all_bottom_with_features = []

for d in eval_dates:
    day_df = val[(val['date'] == d) & (val['ticker'] != 'SPY')]
    if len(day_df) < _TOP_N * 2:
        continue
    all_picks_with_features.append(day_df.nlargest(_TOP_N, 'predicted'))
    all_bottom_with_features.append(day_df.nsmallest(_TOP_N, 'predicted'))

top_df    = pd.concat(all_picks_with_features)
bottom_df = pd.concat(all_bottom_with_features)

profile = pd.DataFrame({
    'Top-20 picks (mean)':    top_df[FEATURE_COLS].mean(),
    'All val stocks (mean)':  val[FEATURE_COLS].mean(),
    'Bottom-20 (mean)':       bottom_df[FEATURE_COLS].mean(),
}).round(3)

print("What kind of stocks does GBM pick vs avoid?")
print()
print(profile.to_string())

In [ ]:
# Visualize the profile differences for key features
key_features = ['pct_sma200', 'pct_ath', 'log_ret_252d', 'vol_20d', 'pct_52w_low', 'pct_time_since_ath']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    bins = 50
    rng = (val[feat].quantile(0.01), val[feat].quantile(0.99))
    ax.hist(val[feat], bins=bins, range=rng, alpha=0.4, color='gray', density=True, label='All val')
    ax.hist(top_df[feat], bins=bins, range=rng, alpha=0.6, color='steelblue', density=True, label='Top-20 picks')
    ax.axvline(val[feat].mean(), color='gray', linestyle='--', linewidth=1)
    ax.axvline(top_df[feat].mean(), color='steelblue', linestyle='--', linewidth=1)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('Feature distributions: GBM top-20 picks vs all val stocks', y=1.01)
plt.tight_layout()
plt.show()

---
## Part 9: Robustness Checks

A few sanity checks to spot potential problems.

In [ ]:
# 1. How often does the same ticker appear across evaluation periods?
# High overlap might mean we're just holding the same concentrated bet
ticker_counts = all_picks_df['ticker'].value_counts()
print(f"Total unique tickers picked across all val periods: {len(ticker_counts)}")
print(f"Evaluation periods: {len(eval_dates)}")
print()
print("Most frequently picked tickers:")
print(ticker_counts.head(20).to_string())
print()
print(f"Tickers picked only once: {(ticker_counts == 1).sum()}")
print(f"Tickers picked 5+ times: {(ticker_counts >= 5).sum()}")

In [ ]:
# 2. Are returns concentrated in a few good periods, or consistent?
print("Excess returns by period (sorted):")
print(results_df['excess'].sort_values().round(4).to_string())
print()
print(f"Periods with positive excess: {(results_df['excess'] > 0).sum()} / {len(results_df)}")
print(f"Best period:  {results_df['excess'].max():.4f} on {results_df['excess'].idxmax().date()}")
print(f"Worst period: {results_df['excess'].min():.4f} on {results_df['excess'].idxmin().date()}")

In [ ]:
# 3. Does model performance degrade over time in the val set?
# If yes, the training signal may be weakening
n = len(results_df)
first_half  = results_df.iloc[:n//2]['excess'].mean()
second_half = results_df.iloc[n//2:]['excess'].mean()

print(f"First half of val period  ({results_df.index[0].date()} - {results_df.index[n//2-1].date()}):")
print(f"  Mean excess return: {first_half:.4f}")
print()
print(f"Second half of val period ({results_df.index[n//2].date()} - {results_df.index[-1].date()}):")
print(f"  Mean excess return: {second_half:.4f}")
print()
if second_half < first_half * 0.5:
    print("⚠️  Performance dropped significantly in the second half — worth monitoring.")
elif second_half > first_half:
    print("✅  Performance improved in the second half — signal may be strengthening.")
else:
    print("✅  Performance roughly consistent across the val period.")

---
## Part 10: Current Picks (Live Inference)

What would the model buy right now, based on the most recent market data?

In [ ]:
from src.selection.base import DataAccessLayer
from src.selection.quant import QuantModel

dal = DataAccessLayer()
quant = QuantModel()

# refresh_prices=False uses the cached price data from the last quant run
holdings = quant.run({'model': 'gbm', 'eval_date': 'today', 'refresh_prices': False}, dal)

print(f"GBM top picks as of most recent data ({len(holdings)} holdings):\n")
for h in holdings:
    print(f"  {h.ticker:<6} conviction={h.conviction:.3f}  predicted={h.metadata.get('predicted_log_ret', 0):+.4f}")
    print(f"         {h.rationale}")
    print()

In [ ]:
# Summarize the feature profile of today's picks
if holdings:
    live_meta = pd.DataFrame([h.metadata for h in holdings], index=[h.ticker for h in holdings])
    feat_cols_available = [c for c in FEATURE_COLS if c in live_meta.columns]
    print("Feature profile of today's picks:")
    print(live_meta[feat_cols_available + ['predicted_log_ret']].round(3).to_string())